# Task console quickstart — 跑一次完整读出实验

这个 notebook 教你用 **task 控制台**(`exp.task_console()`)跑一次中性原子装载读出实验:
拍相机活图 → 校准读出 → 逐发判 occupancy → 看每个 site 的装载。控制台是五层结构,
各层只经 `SignalHub` 信号耦合,互不直接依赖:

| 层 | 是什么 | 例子 |
|---|---|---|
| **device** | 硬件本体(相机 / sequencer) | `exp.camera`、`exp.devices.sequencer` |
| **measurement** | 持续采集、往 hub 发信号的节点 | Camera(发 `frame`)、Temperature、Fidelity vs duration |
| **processor** | 反应式地消费信号、再发信号 | Judge occupancy(消费 `frame` → 发 `occupied`/`counts`/…) |
| **task** | 一次性编排、直接驱动 device | Calibrate readout(拍模板帧 → 出标定) |
| **plot** | 纯视图,只读一个 hub 信号画图 | 2D 相机图、site map、monitor、hist |

> **虚拟 == 实机。** 下面每一步都是真机要跑的流程;**唯一虚拟的是相机帧**。换真机只改第一格
> `na.connect("virtual", …)` → `na.connect("qcmos"/"remote_template", …, open_devices=True)`,
> 控制台调用、measurement/processor/task 一行都不用动(最后一节给真机配方)。

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = PROJECT_ROOT

import Zou_lab_control.frontend as zf

zf.notebook_setup()

In [ ]:
import numpy as np

import Zou_lab_control.frontend as zf
import Zou_lab_control.neutral_atom as na
# 控制台用一个 SignalHub 做层间总线;它不在 na 顶层,从 core.signals 取。
from Zou_lab_control.neutral_atom.core.signals import SignalHub

try:
    zf.use_widget_backend()
except Exception as exc:
    print(f"Widget backend not enabled here: {exc}")

zf.enable_long_output()
zf.apply_style()

## 1. 连接 device

虚拟会话:相机帧仿真,其余(标定、判 occupancy、脉冲编译)走和真机完全相同的契约路径。
`sitemap` 给出 trap 网格形状等,虚拟相机据此造帧。

In [ ]:
exp = na.connect(
    "virtual",
    sitemap={"grid_shape": (5, 7), "spacing_px": 12.0, "roi_radius": 1},
)
exp

## 2. 控制台能放哪些节点(自动发现的目录)

measurement / processor / task 三个目录都是**自动发现**的:`operations/measurements`、
`operations/processors`、`operations/tasks` 里每个 `@measurement`/`@processor`/`@task` 工厂
(以及 notebook 里 `register_*` 注册的)都会出现在这里,也出现在控制台头部 **Add Panel** 下拉里。
每个 spec 把自己的参数声明一次(GUI 表单和 API 默认值同源,不会漂移)。

In [ ]:
print("measurements:", [s.name for s in exp.readout.measurement_specs()])
print("processors  :", [s.name for s in exp.readout.processor_specs()])
print("tasks       :", [s.name for s in exp.readout.task_specs()])

## 3. 打开控制台

一行打开:**`exp.task_console()`**——它已经把会话、三个自动发现的目录(measurement / processor / task)
和一个内部 `SignalHub` 都接好了,你**不用手写任何 hub / 目录 / 布局配置**。`on_close=exp.close` 让关窗时
一并安全断开 device(关窗会停刷新 timer + 每个在跑节点的采集线程,真正释放相机)。

在桌面 Qt 环境里取消下面的注释运行;无显示环境(本 notebook 自动跑批)就跳过——
第 5 节用**纯 API**走同一条路,不需要 GUI。

> **面板怎么选信号(下拉 = 表达式的快捷方式)**:每个 plot 面板的 Setting 里有一个 **signal 下拉**
> (列出所有 live 信号,按**产出节点分组**,信号名就是它在 hub 上的真名,如 `rate`/`occupied`/`frame`)。
> 选一个 = 把面板的 **source** 设成 `value = signal`(最常用)。source 本身是**一行可编辑的 Python 表达式**:
> 除了 `value = signal`,你也能直接点名某信号(`value = rate`)或跨信号运算(如两块占据相减 `value = occupied - b_occupied`)。
> 下拉是快捷方式,表达式是底层——两者是同一个东西。

**Add Panel 实验流程**(头部下拉选一项 → Add Panel):

1. **Camera(measurement)** → 它的 Edit 页是相机参数(`exposure` 曝光、`region` ROI、
   `frames_per_cycle`)。点 **Start**:开始往 hub 发 `frame`。
2. **2D plot** → Setting 里 signal 选 `frame` → 看活相机图。
3. **Task: Calibrate readout** → Edit 页(每个字段都有真默认值,没有空白):
   - `source` = **live**(现在拍) / **saved frames**(读盘上已有的原始帧);复用一份已标好的标定不在这里——直接让 Judge-occupancy 指它的 `calibration.json`;
   - `folder`:**Browse** 选数据/报告文件夹(live 跑完把标定 + 报告写进 folder **根目录**(原始帧另进干净的 `frames/` 子目录):**每种读出方式各一张逐站点分布图**(`site_distribution_box.png` / `_psf.png` / `_uniform_psf.png`)+ 全局分布 + site map + **`psf_table.csv` / `psf_grid.png`**(逐站点真实非对称 PSF 形状);报告图都经 `data_figure` 的统一存盘 dpi(600)存清晰,实时面板则按 150 dpi 渲染提速、显示尺寸不变);
   - `pulse template`:**Browse** 选成像脉冲程序(默认 `imaging_template.json`)——它**本身就是 long-short-long bracket**(`load` + 三个 emCCD 帧 `image_0/1/2`,帧间只 hold trap 的 gap、不重做 cooling);cali **不派生 bracket**,只用下面两个曝光按名设它的长/短帧 duration,所以 **file == fired**(在 pulse GUI 打开这个模板看到的就是 cali 实际发的脉冲);
   - `reference exposure`(**长**参考帧,两端,写 API slot `a1`)/ `readout exposure`(**短**读出帧,中间,学阈值的时长,写 API slot `a2`)——**这两个值直接在这里设**;`threshold` = otsu / bimodal;`reference brackets`(发数)/ `ROI radius`。
   - 点 **Start**:task 接管控制台(confocal 式锁定),把模板帧流到专属 Monitor 面板,跑完
     **自动成为会话标定**。注意:**启动 task 会先停掉相机等其它在跑节点**(task 直接驱动相机+sequencer,
     否则抢资源会卡死);跑完再回去 Start 相机。
4. **Judge occupancy(processor)** → Edit 页:
   - `calibration`:**Browse** 一个存盘标定;**留空 = 用刚标好的会话标定**;
   - `source`:hub 信号下拉,选 `frame`(像 plot 的入口那样选信号,不是手打表达式);
   - `method`:**box / per-site PSF / uniform PSF**(读出方式在这里选——标定一次就把三种都算好了);`ema`。
   - 点 Start:发 `occupied`/`counts`/`rate`/`centers`/`frame_judged` 等(就是这些短名字,不带节点前缀)。
5. **Site map(plot)** → Setting 里**只选一个信号** `occupied`:圆心和帧底图从**同一个产出节点**
   自动解析(环和底图永远是同一发),不用再分别指定 centers/image。
6. 再加 **monitor**(`value = rate` 看装载率滚动)/ **hist**(`value = counts`)等视图;每张面板的
   Setting 里还能单独设它的 **update**(刷新率,如做 alignment 监视的快 100 ms),Edit 页有 colormap /
   relim / unit / 拟合 / 命令框 / 保存。

> **API slot(`a1`/`a2`...)**:pulse 编辑器里点一个 duration/DAC 格子的圆点,会在 **none → scan(`sN`,橙) → API(`aN`,紫,数字保留) → none** 之间循环。scan 是硬件扫描句柄(值进扫描表);**API slot 是给这个字段起的命名句柄**,notebook/API/task 可以按名直接设值(`pulse.a1 = 1e-3` 或 `state.set_api("a1", v)`),不用 parse 哪个 period 的哪个信号。cali 正是用 `a1`/`a2` 设成像模板的两个曝光。

In [ ]:
# 桌面 Qt 环境里取消注释:直接开控制台,按上面的 Add Panel 流程自己搭。exp.task_console() 已把
# 本 session 的 hub / 测量目录 / 处理器目录 / 任务目录都自动接好——无需手写任何配置代码。
# console = exp.task_console(on_close=exp.close)   # 关窗 = 停所有节点线程 + 安全断开 device
# console

## 4. 也可以一行加载你存好的面板布局

控制台默认开**空板**(没有预设)。你在 GUI 里摆好的板可以 Save 成 `tasks/<name>.json`,
下次用 `task="<name>"` 或 JSON 路径一键复原(同样把相机/标定/判 occupancy 接好)。

```python
console = exp.task_console(task="loading_readout", on_close=exp.close)
# task="loading_readout" -> tasks/loading_readout.json,或直接传一个 .json 路径
```

## 5. 同一条路,不开 GUI:纯 notebook API

控制台的 Add Panel 只是这些节点的 GUI 外壳——measurement/processor/task **在 notebook 里直接调
也完全一样**(同一个建器,GUI 和 API 不会漂移)。下面不开窗,纯脚本跑完整读出:

1. **Calibrate readout task**:`exp.readout.calibrate_task(hub, …)` 建出和 Add-Panel 同款 task,
   `run_to_completion()` 一次拍模板+定阈值,标定自动成为会话标定(`exp.readout.current`)——
   标定一次就带 box / per-site PSF / uniform PSF 全部方法。
2. **Camera measurement + Judge occupancy processor**:`camera.step()` 发 `frame`,
   `detector.step()` 消费 `frame` 发 occupancy。这里手动 `step()` 是为了 notebook 里确定性;
   控制台/真机用 `node.start(rate_hz=…)` 起后台线程。

判 occupancy 用 `frame_judged`(processor 判的**那一发**),所以 `occupied` 和底图永远对齐。

In [ ]:
from Zou_lab_control.neutral_atom.operations.logic import OccupancyProcessor

hub = SignalHub()

# (1) 校准读出 TASK —— 和控制台 Add Panel "Task: Calibrate readout" 同一个建器。
#     live = 现在拍;folder 收标定+报告。pulse_template 文件本身就是 long-short-long bracket
#     (load + 三个 emCCD 帧 image_0/1/2,帧间只 hold trap 的 gap、不重做 cooling)。cali 不派生
#     bracket——它只 load 模板、用 API slot 按名设那两个曝光:reference_exposure 写 a1(两端长帧,
#     投真值+建 site map/PSF)、readout_exposure 写 a2(中间短帧,学阈值)。file == fired:在 pulse
#     GUI 打开这个模板看到的就是 cali 实际发的脉冲。就是 rb87 readout。
task = exp.readout.calibrate_task(
    hub, source="live", threshold_method="otsu",
    pulse_template="imaging_template.json",
    reference_exposure=0.020,       # 长参考帧(两端)-> API slot a1
    readout_exposure=0.005,         # 短读出帧(中间),真正学阈值的时长 -> API slot a2
    threshold_frames=60, folder="results/cal_run",
)
task.run_to_completion()
cal = exp.readout.current          # 跑完自动成为会话标定
print("calibration methods:", cal.methods(), "| n_sites:", len(cal.centers))
# 标定产物落在 results/cal_run/:calibration.json + 每方式分布图 + site_map.png +
# psf_table.csv + psf_grid.png(逐站点的真实非对称 PSF 形状)+ calibration.npz(含 psf_weights)。
# save_frames 默认开,所以原始帧也落进 results/cal_run/frames/:每张 emCCD bracket 帧 img<n>.npy(往返数据)+ 配对的 img<n>.png(走和控制台同一个 2D-image plot type 画的图,肉眼核对用)+ run_schema.json;以后 source="saved frames" 指向同一 folder 复算只读 .npy。

In [ ]:
# (2) 相机 measurement + 反应式 Judge-occupancy processor,作为 logic 节点。

# 相机由脉冲门控:先点亮连续成像脉冲(On Pulse),相机才有帧(同真机;Stop Pulse 即冻结)。
live_pulse = exp.timing.bind_pulse(na.imaging_sequence(exposure=exp.camera.exposure, load=True))
live_pulse.on_pulse(wait=False, repeat_forever=True)
camera = exp.readout.camera_measurement(hub)       # 发 'frame'
detector = OccupancyProcessor(                      # 消费 'frame' -> occupancy
    hub, calibration=cal, source_expr={"inputs": ["frame"], "source": "value = signal"},
    method="box",                                  # 'box' | 'psf' | 'uniform_psf' —— 在这里选读出方式
    grid_shape=exp.devices.trap_array.grid_shape,
)

for _ in range(8):          # notebook 里确定性步进;真机/GUI 用 node.start(rate_hz=...)
    camera.step()
    detector.step()

occupied = hub.latest("occupied")[-1]   # 取最近一发:occupied 是 (repeat, n_sites) 块,只看最后一发 -> (n_sites,)
print("loaded sites:", int(occupied.sum()), "/", occupied.size)
# site map 的圆心+底图都来自这同一个节点:occupied 与 detect(frame_judged) 一致 == 不会错位。
frame_judged = hub.latest("frame_judged")[-1]   # frame_judged 是 (repeat, H, W) 块;detect 吃单帧,取最后一发 (H, W)
print("occupied == detect(frame_judged):",
      bool(np.array_equal(occupied, cal.detect(frame_judged, method="box").occupied)))

In [ ]:
# occupancy 网格:把 1D occupied 重排成 trap 网格,直接拿去做统计。
occupied.reshape(exp.devices.trap_array.grid_shape).astype(int)

## 6. 换真机(只改连接那一行)

先在 FPGA/Vivado 电脑上起 sequencer server(见 `neutral_atom_hardware_quickstart.ipynb`),
再把第 1 格换成真机连接,**本 notebook 其余每一格不动**:

```python
exp = na.connect(
    "remote_template",                       # 或 "qcmos" 等你的设备图
    sequencer={"host": "192.168.0.20", "port": 18861},
    open_devices=True,
)
```

- **Calibrate readout** 的 `pulse template` 选你自己的成像程序(pulse GUI 存的 `PulseTableState .json`,
  比如 `pulses/camera_imaging_address_switch.json`);它**本身就是 long-short-long bracket**(三个 emCCD
  帧、帧间只 hold trap)。cali 只用 `reference_exposure`(长)+ `readout_exposure`(短)按名(API slot
  `a1`/`a2`)设那两个曝光 duration——**file == fired**,绝不另外派生脉冲。
- task 在真机上**直接 on_pulse 这个模板并逐发读相机**,所发 timing 会被 server 记录;之后在 pulse GUI
  按 **Sync** 就能把刚跑的 long-short-long 拉回编辑器看(GUI 和 API 背后是同一份 PulseTableState)。
- 控制台关窗会停所有采集线程并(配 `on_close=exp.close`)安全断开 device——不会有线程占着相机/RPyC。
- 真机上判 occupancy 仍走 `calibration.detect` 同一契约;换的只有相机帧来源。

## 7. 存标定 / 收尾

标定可存盘供下次 Judge-occupancy 直接 Browse 加载;notebook 收尾时把会话关掉释放 device。

In [ ]:
from pathlib import Path
Path("results").mkdir(exist_ok=True)
calibration_path = exp.readout.save("results/task_console_calibration.json")
print("saved:", calibration_path)

exp.close()   # 释放 device(GUI 路径里这步由 on_close=exp.close 自动完成)